In [1]:

import json
import pickle
import time
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

start = time.time()

# ----------------------------
# 0) Project / MLflow setup
# ----------------------------
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
PREP_ROOT = DATA_ROOT / "prepared"
REPORT_ROOT = PROJECT_ROOT / "reports"
ARTIFACT_ROOT = PROJECT_ROOT / "models" / "artifacts"
MLRUNS_ROOT = PROJECT_ROOT / "mlruns"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MLRUNS_ROOT.mkdir(parents=True, exist_ok=True)

tracking_uri = f"file:///{MLRUNS_ROOT.resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("recomart_content_based")

INTERACTIONS_PATH = PREP_ROOT / "interactions_prepared.csv"
PRODUCTS_PATH = PREP_ROOT / "products_prepared.csv"

# ----------------------------
# 1) Load data
# ----------------------------
interactions = pd.read_csv(INTERACTIONS_PATH)
products = pd.read_csv(PRODUCTS_PATH)

interactions.columns = [c.strip().lower() for c in interactions.columns]
products.columns = [c.strip().lower() for c in products.columns]

required_interaction_cols = {"user_id", "item_id", "event_weight"}
missing_i = required_interaction_cols - set(interactions.columns)
if missing_i:
    raise ValueError(f"Missing required interaction columns: {missing_i}")

# flexible product id detection
product_id_col = None
for c in ["product_id", "item_id", "id"]:
    if c in products.columns:
        product_id_col = c
        break

if product_id_col is None:
    raise ValueError("Could not find a product ID column in products_prepared.csv")

# normalize IDs to avoid int/str mismatches
interactions["user_id"] = interactions["user_id"].astype(str).str.strip()
interactions["item_id"] = interactions["item_id"].astype(str).str.strip()
products[product_id_col] = products[product_id_col].astype(str).str.strip()

# numeric weight cleanup
interactions["event_weight"] = pd.to_numeric(interactions["event_weight"], errors="coerce").fillna(0.0)

print("Loaded interactions:", interactions.shape)
print("Loaded products:", products.shape)
print("Using product ID column:", product_id_col)

# ----------------------------
# 2) Aggregate interactions
# ----------------------------
interactions = (
    interactions.groupby(["user_id", "item_id"], as_index=False)["event_weight"]
    .sum()
)

# ----------------------------
# 3) Prepare product content features
# ----------------------------
products = products.drop_duplicates(subset=[product_id_col]).copy()

candidate_text_cols = [
    "title", "name", "product_name", "category", "brand",
    "description", "desc", "subcategory", "color"
]
available_text_cols = [c for c in candidate_text_cols if c in products.columns]

if not available_text_cols:
    raise ValueError("No usable text columns found in products_prepared.csv for content-based features.")

for c in available_text_cols:
    products[c] = products[c].fillna("").astype(str).str.lower().str.strip()

# optional numeric feature: price bucket
if "price" in products.columns:
    products["price"] = pd.to_numeric(products["price"], errors="coerce")
    non_null_price = products["price"].notna().sum()
    if non_null_price >= 5:
        products["price_bucket"] = pd.qcut(
            products["price"].rank(method="first"),
            q=min(5, products["price"].nunique()),
            labels=False,
            duplicates="drop"
        ).astype("Int64").astype(str)
    else:
        products["price_bucket"] = ""
else:
    products["price_bucket"] = ""

products["content_text"] = products[available_text_cols].agg(" ".join, axis=1).str.strip()
products["content_text"] = (
    products["content_text"] + " pricebucket_" + products["price_bucket"].fillna("")
).str.strip()

products = products[[product_id_col, "content_text"] + [c for c in available_text_cols if c not in [product_id_col]]].copy()
products = products[products["content_text"].str.len() > 0].copy()

# ----------------------------
# 4) OVERLAP FILTER FIRST (critical fix)
# ----------------------------
catalog_item_ids = set(products[product_id_col])
overlap_items = set(interactions["item_id"]) & catalog_item_ids

print("\nOverlap diagnostics")
print("Total unique interaction items:", interactions["item_id"].nunique())
print("Total catalog items:", len(catalog_item_ids))
print("Overlapping items:", len(overlap_items))

interactions_cb = interactions[interactions["item_id"].isin(overlap_items)].copy()
products_cb = products[products[product_id_col].isin(overlap_items)].copy()

print("Rows after item-overlap filter:", len(interactions_cb))
print("Users after item-overlap filter:", interactions_cb["user_id"].nunique())
print("Items after item-overlap filter:", interactions_cb["item_id"].nunique())

# Keep only users with enough overlap for leave-one-out
user_item_counts = interactions_cb.groupby("user_id")["item_id"].nunique()
eligible_users = user_item_counts[user_item_counts >= 2].index
interactions_cb = interactions_cb[interactions_cb["user_id"].isin(eligible_users)].copy()

print("Eligible users with >=2 overlapped items:", interactions_cb["user_id"].nunique())

if interactions_cb.empty:
    raise ValueError("No usable overlap remains after filtering. Expand product coverage or inspect item ID mapping.")

# ----------------------------
# 5) Leave-one-out split AFTER overlap filtering
# ----------------------------
rng = np.random.RandomState(42)
interactions_cb["_rand"] = rng.rand(len(interactions_cb))
test_idx = interactions_cb.groupby("user_id")["_rand"].idxmin()

test_df = interactions_cb.loc[test_idx, ["user_id", "item_id", "event_weight"]].copy()
train_df = interactions_cb.drop(index=test_idx).copy()

# safety check: keep only users with at least 1 training item
train_counts = train_df.groupby("user_id")["item_id"].nunique()
valid_users = train_counts[train_counts >= 1].index
train_df = train_df[train_df["user_id"].isin(valid_users)].copy()
test_df = test_df[test_df["user_id"].isin(valid_users)].copy()

print("\nSplit diagnostics")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train users:", train_df["user_id"].nunique())
print("Test users:", test_df["user_id"].nunique())

if test_df.empty:
    raise ValueError("Test set is empty after filtering. Cannot evaluate metrics.")

# ----------------------------
# 6) Build TF-IDF matrix
# ----------------------------
products_cb = products_cb.drop_duplicates(subset=[product_id_col]).copy()
products_cb = products_cb.sort_values(product_id_col).reset_index(drop=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=1,
    ngram_range=(1, 2)
)
tfidf_matrix = vectorizer.fit_transform(products_cb["content_text"])

item_ids = products_cb[product_id_col].astype(str).tolist()
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
idx_to_item = np.array(item_ids)

print("TF-IDF matrix:", tfidf_matrix.shape)
print("Feature vocab size:", len(vectorizer.vocabulary_))

# ----------------------------
# 7) Build user history
# ----------------------------
train_df = train_df[train_df["item_id"].isin(item_to_idx)].copy()
test_df = test_df[test_df["item_id"].isin(item_to_idx)].copy()

if test_df.empty:
    raise ValueError("All test items were removed after TF-IDF item alignment.")

user_history = (
    train_df.groupby("user_id")[["item_id", "event_weight"]]
    .apply(lambda g: list(zip(g["item_id"].tolist(), g["event_weight"].tolist())))
    .to_dict()
)

# ----------------------------
# 8) Recommendation function
# ----------------------------
def recommend_content(user_id, k=10):
    if user_id not in user_history:
        return []

    history = user_history[user_id]
    valid_history = [(item, wt) for item, wt in history if item in item_to_idx]

    if not valid_history:
        return []

    seen_item_ids = [item for item, _ in valid_history]
    seen_indices = np.array([item_to_idx[item] for item in seen_item_ids], dtype=np.int32)
    seen_weights = np.array([float(wt) for _, wt in valid_history], dtype=np.float32)

    # weighted user profile
    user_profile = np.average(tfidf_matrix[seen_indices].toarray(), axis=0, weights=seen_weights)
    scores = linear_kernel([user_profile], tfidf_matrix).ravel()

    # remove seen items
    scores[seen_indices] = -np.inf

    finite_mask = np.isfinite(scores)
    candidate_count = int(finite_mask.sum())
    if candidate_count == 0:
        return []

    top_n = min(k, candidate_count)
    top_idx = np.argpartition(scores, -top_n)[-top_n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    top_idx = top_idx[np.isfinite(scores[top_idx])]

    return idx_to_item[top_idx].tolist()

sample_user = train_df["user_id"].iloc[0]
sample_recs = recommend_content(sample_user, k=5)
print("\nSample user:", sample_user)
print("Top-5 recommendations:", sample_recs)

# ----------------------------
# 9) Evaluation
# ----------------------------
def precision_recall_ndcg_at_k(eval_test_df, k=10, max_eval_users=10000):
    user_truth = eval_test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    eval_users = list(user_truth.keys())[:max_eval_users]

    precisions, recalls, ndcgs = [], [], []

    for user_id in eval_users:
        recs = recommend_content(user_id, k=k)
        if not recs:
            continue

        true_items = user_truth[user_id]
        hits = np.array([1.0 if item in true_items else 0.0 for item in recs], dtype=np.float32)

        precision = float(hits.sum() / k)
        recall = float(hits.sum() / len(true_items))

        discounts = 1.0 / np.log2(np.arange(2, len(hits) + 2))
        dcg = float((hits * discounts).sum())

        ideal_len = min(len(true_items), k)
        idcg = float((np.ones(ideal_len) / np.log2(np.arange(2, ideal_len + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        ndcgs.append(ndcg)

    return {
        f"precision_at_{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall_at_{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"ndcg_at_{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"evaluated_users_at_{k}": int(len(precisions)),
    }

metrics_5 = precision_recall_ndcg_at_k(test_df, k=5, max_eval_users=10000)
metrics_10 = precision_recall_ndcg_at_k(test_df, k=10, max_eval_users=10000)

runtime_minutes = (time.time() - start) / 60.0

all_metrics = {
    **metrics_5,
    **metrics_10,
    "runtime_minutes": float(runtime_minutes),
    "catalog_size_used": int(len(products_cb)),
    "feature_vocab_size": int(len(vectorizer.vocabulary_)),
    "overlap_items": int(len(overlap_items)),
    "eligible_users": int(train_df["user_id"].nunique()),
}

print("\nContent-Based Metrics")
for metric, value in all_metrics.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

# ----------------------------
# 10) Save artifacts
# ----------------------------
metrics_path = REPORT_ROOT / "content_based_metrics.json"
summary_path = REPORT_ROOT / "content_based_run_summary.json"
bundle_path = ARTIFACT_ROOT / "content_based_bundle.pkl"

run_summary = {
    "input_files": {
        "interactions": str(INTERACTIONS_PATH),
        "products": str(PRODUCTS_PATH),
    },
    "product_id_column": product_id_col,
    "text_columns_used": available_text_cols,
    "train_shape": list(train_df.shape),
    "test_shape": list(test_df.shape),
    "tfidf_shape": list(tfidf_matrix.shape),
    "sample_user": str(sample_user),
    "sample_top5_recommendations": [str(x) for x in sample_recs],
    "metrics": all_metrics,
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

with open(bundle_path, "wb") as f:
    pickle.dump(
        {
            "vectorizer": vectorizer,
            "tfidf_matrix": tfidf_matrix,
            "item_to_idx": item_to_idx,
            "idx_to_item": idx_to_item,
            "user_history": user_history,
            "product_id_column": product_id_col,
            "text_columns_used": available_text_cols,
        },
        f,
    )

# ----------------------------
# 11) MLflow logging
# ----------------------------
with mlflow.start_run(run_name="content_based_baseline_fixed_overlap") as run:
    mlflow.log_param("model_type", "content_based_filtering")
    mlflow.log_param("feature_method", "tfidf_text_plus_optional_price_bucket")
    mlflow.log_param("interactions_file", str(INTERACTIONS_PATH))
    mlflow.log_param("products_file", str(PRODUCTS_PATH))
    mlflow.log_param("product_id_column", product_id_col)
    mlflow.log_param("text_columns_used", ",".join(available_text_cols))
    mlflow.log_param("split_type", "leave_one_out_after_catalog_overlap_filter")
    mlflow.log_param("ngram_range", "1_2")
    mlflow.log_param("stop_words", "english")

    mlflow.log_param("train_rows", int(len(train_df)))
    mlflow.log_param("test_rows", int(len(test_df)))
    mlflow.log_param("train_users", int(train_df["user_id"].nunique()))
    mlflow.log_param("train_items", int(train_df["item_id"].nunique()))
    mlflow.log_param("catalog_size_used", int(len(products_cb)))
    mlflow.log_param("feature_vocab_size", int(len(vectorizer.vocabulary_)))
    mlflow.log_param("overlap_items", int(len(overlap_items)))

    mlflow.log_metrics(all_metrics)
    mlflow.log_artifact(str(metrics_path))
    mlflow.log_artifact(str(summary_path))
    mlflow.log_artifact(str(bundle_path))

    run_id = run.info.run_id

print("\nMLflow tracking URI:", tracking_uri)
print("MLflow run_id:", run_id)
print("Saved artifacts:")
print("-", metrics_path)
print("-", summary_path)
print("-", bundle_path)


C:\Users\barath\AppData\Local\anaconda3\envs\dm4ml\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Loaded interactions: (2755641, 10)
Loaded products: (194, 100)
Using product ID column: product_id



Overlap diagnostics


Total unique interaction items: 235061
Total catalog items: 194
Overlapping items: 92
Rows after item-overlap filter: 667
Users after item-overlap filter: 663
Items after item-overlap filter: 92
Eligible users with >=2 overlapped items: 3

Split diagnostics
Train shape: (4, 4)
Test shape: (3, 3)
Train users: 3
Test users: 3
TF-IDF matrix: (92, 557)
Feature vocab size: 557

Sample user: 1150086
Top-5 recommendations: ['26', '16', '33', '32', '40']

Content-Based Metrics
precision_at_5: 0.0000
recall_at_5: 0.0000
ndcg_at_5: 0.0000
evaluated_users_at_5: 3
precision_at_10: 0.0000
recall_at_10: 0.0000
ndcg_at_10: 0.0000
evaluated_users_at_10: 3
runtime_minutes: 0.2444
catalog_size_used: 92
feature_vocab_size: 557
overlap_items: 92
eligible_users: 3



MLflow tracking URI: file:///C:/Users/barath/recomart-pipeline/mlruns
MLflow run_id: 4f63e8f2c0c94d7e88e88114054e8426
Saved artifacts:
- C:\Users\barath\recomart-pipeline\reports\content_based_metrics.json
- C:\Users\barath\recomart-pipeline\reports\content_based_run_summary.json
- C:\Users\barath\recomart-pipeline\models\artifacts\content_based_bundle.pkl
